# Transformers (self-attention, multi-head) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## Stack self-attention, residual streams, normalization and MLPs into a parallel sequence model
The examples isolate self-attention weights, multi-head diversity, residual addition, layer normalization and a block-shaped end-to-end pass.

In [ ]:
X=np.array([[1.,0.],[0.,1.],[1.,1.]]); S=X@X.T/np.sqrt(2); A=softmax(S,axis=1)
plt.figure(figsize=(4,3)); plt.imshow(A,cmap='Blues'); plt.colorbar(); plt.title('Self-attention: every token attends to every token')
assert A.shape==(3,3) and np.allclose(A.sum(1),1)

In [ ]:
A1=softmax(np.array([[2,1,0],[0,2,1],[1,0,2.]],float),axis=1); A2=softmax(np.array([[0,1,2],[2,0,1],[1,2,0.]],float),axis=1)
plt.figure(figsize=(7,3)); plt.subplot(1,2,1); plt.imshow(A1); plt.title('head 1'); plt.subplot(1,2,2); plt.imshow(A2); plt.title('head 2')
assert not np.allclose(A1,A2)

In [ ]:
x=np.array([1.,2.]); attn=np.array([0.5,-0.5]); y=x+attn
plt.figure(figsize=(4,3)); plt.bar(['x0','x1'],x,label='residual'); plt.bar(['x0','x1'],attn,bottom=x,label='attention'); plt.legend(); plt.title('Residual path preserves original signal')
assert np.allclose(y,[1.5,1.5])

In [ ]:
z=np.array([1.,2.,5.]); ln=(z-z.mean())/z.std()
plt.figure(figsize=(5,3)); plt.bar(['raw1','raw2','raw3'],z,alpha=.5,label='raw'); plt.plot(range(3),ln,'-o',label='LayerNorm'); plt.legend(); plt.title('LayerNorm centers and scales features')
assert abs(ln.mean())<1e-9 and abs(ln.std()-1)<1e-9

In [ ]:
X=np.array([[1.,0.],[0.,1.],[1.,1.]]); A=softmax(X@X.T/np.sqrt(2),axis=1); H=A@X; Y=X+H; Y=(Y-Y.mean(1,keepdims=True))/(Y.std(1,keepdims=True)+1e-9)
plt.figure(figsize=(5,3)); plt.imshow(Y,cmap='coolwarm'); plt.colorbar(); plt.title('One tiny Transformer block output')
assert Y.shape==X.shape and np.allclose(Y.mean(1),0)